# INFO-6147-(01)-26W Deep Learning with Pytorch

## Project:

**Student Name:** Yun-Jiung Wang

**Student Number:** 1256222

**Date:** March 30th, 2026

**Description:**

This project is aim to allow user upload the photos of food and describe what the food tastes like. It will be helpful when traveling or when people wants to try international food, but not having a person to explain to you what that is.
Here comes this AI project, to assist people on observing new foods.

## Setup Env

In [ ]:
!pip install datasets gradio streamlit groq pyngrok

# dlownload Cloudflare tunnle tool
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

## Import Libs

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import random_split,DataLoader,Subset

from datasets import load_dataset

import seaborn as sns
from sklearn.metrics import confusion_matrix, f1_score

import io
from tqdm import tqdm
import streamlit as st
from groq import Groq

from google.colab import userdata,runtime,drive # Store the API via Colab Secret
import zipfile
import os


## Check Device

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## Setup Google Drive

In [ ]:
drive.mount('/content/drive')


DRIVE_BASE = '/content/drive/MyDrive/Colab/AIM1_level02/INFO-6147-DL With Pytorch/data/'
os.makedirs(DRIVE_BASE, exist_ok=True)

ZIP_PATH = os.path.join(DRIVE_BASE, 'food_101.zip')
LOCAL_DATA_PATH = '/content/food_data'

if os.path.exists(ZIP_PATH) and not os.path.exists(LOCAL_DATA_PATH):
    import zipfile
    print("Unzipping data to local machin...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall('/content/')
    print("Unzipping complete")

## Setup Hyperparameters

In [ ]:
EPOCHS=5
BATCH_SIZE = 256
LEARNING_RATE=0.001
NUM_WORKERS= 4

DATA_PATH = './data'

## Load Dataset (Colab)

In [ ]:
# Transformer

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Load dataset (train and test)
train_dataset = datasets.Food101(root=DATA_PATH, split='train', download=True, transform=train_transform)

test_dataset = datasets.Food101(root=DATA_PATH, split='test', download=True, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)

test_loader = DataLoader(test_dataset,
                         batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

num_classes = len(train_dataset.classes)
print(f"Load Food101 Successfully！Num of Classes: {num_classes}")

## Build the model

In [ ]:
# model = models.resnet50(pretrained=True)

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

model.fc = nn.Sequential(
    nn.Linear(model.fc.in_features, 512),
    nn.ReLU(),                # activate param
    nn.Dropout(0.5),          # drop off 50%
    nn.Linear(512, num_classes) # output classes
)

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE,weight_decay=1e-4)

## Define the Model

In [ ]:
from google.colab import runtime

class FoodClassifier:
    def __init__(self, model, train_loader, test_loader, criterion, optimizer, device):

        self.model = model.to(device)
        self.train_loader = train_loader
        self.test_loader = test_loader
        self.criterion = criterion
        self.optimizer = optimizer
        self.device = device
        # The class still holds the data, but won't "show" it
        self.history = {
            'train_loss': [], 'train_acc': [],
            'test_loss': [], 'test_acc': []
        }

    def train_epoch(self):
        self.model.train()
        running_loss, correct, total = 0.0, 0, 0
        pbar = tqdm(self.train_loader, desc="[Train]", leave=False)

        for images, labels in pbar:
            images, labels = images.to(self.device), labels.to(self.device)
            self.optimizer.zero_grad()
            outputs = self.model(images)
            loss = self.criterion(outputs, labels)
            loss.backward()
            self.optimizer.step()

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        return running_loss / len(self.train_loader), 100. * correct / total

    def validate(self):
        self.model.eval()
        running_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in self.test_loader:
                images, labels = images.to(self.device), labels.to(self.device)
                outputs = self.model(images)
                loss = self.criterion(outputs, labels)
                running_loss += loss.item()
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()

        return running_loss / len(self.test_loader), 100. * correct / total

    def save_checkpoint(self, filename="food_model.pth"):
      """Stores model weights"""
      torch.save(self.model.state_dict(), filename)
      print(f"💾 Model weights saved to {filename}")

    def run_training(self, num_epochs, save_path):
        for epoch in range(num_epochs):
            t_loss, t_acc = self.train_epoch()
            v_loss, v_acc = self.validate()
            self.history['train_loss'].append(t_loss)
            self.history['train_acc'].append(t_acc)
            self.history['test_loss'].append(v_loss)
            self.history['test_acc'].append(v_acc)
            print(f"Epoch {epoch+1}/{num_epochs} | Train Acc: {t_acc:.2f}% | Test Acc: {v_acc:.2f}%")

        # saved before leave
        self.save_checkpoint(os.path.join(save_path, "food_model.pth"))
        print("🌙 releasing A100 points...")
        runtime.unassign()

    def get_history(self):
      """Returns training history for external plotting"""
      return self.history

## Visualize Loss and Accuracy Curve

In [ ]:
def plot_training_results(history):
    epochs = range(1, len(history['train_loss']) + 1)
    plt.figure(figsize=(14, 5))

    # Loss
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_loss'], 'b-o', label='Training Loss')
    plt.plot(epochs, history['test_loss'], 'r--s', label='Validation Loss')
    plt.title('Model Convergence (Loss)')
    plt.xlabel('Epochs')
    plt.ylabel('Loss Value')
    plt.legend()
    plt.grid(True, linestyle=':', alpha=0.6)

    # Accuracy
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['train_acc'], 'g-o', label='Training Acc')
    plt.plot(epochs, history['test_acc'], 'm--s', label='Validation Acc')
    plt.title('Model Performance (Accuracy)')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.grid(True, linestyle=':', alpha=0.6)

    plt.tight_layout()
    plt.show()

## Define result visualization function

In [ ]:
def visualize_results(model, dataset, num_images=6):
    model.eval()
    fig = plt.figure(figsize=(15, 10))
    class_names = dataset.classes

    # select randomly
    indices = np.random.choice(len(dataset), num_images, replace=False)

    with torch.no_grad():
        for i, idx in enumerate(indices):
            image, label = dataset[idx]
            input_tensor = image.unsqueeze(0).to(device)

            output = model(input_tensor)
            _, pred = torch.max(output, 1)

            # transfer into showable picture
            img_display = image.permute(1, 2, 0).numpy()
            mean = np.array([0.485, 0.456, 0.406])
            std = np.array([0.229, 0.224, 0.225])
            img_display = std * img_display + mean
            img_display = np.clip(img_display, 0, 1)

            ax = plt.subplot(2, 3, i + 1)
            color = 'green' if pred.item() == label else 'red'
            ax.set_title(f"Pred: {class_names[pred.item()]}\nActual: {class_names[label]}", color=color)
            plt.imshow(img_display)
            plt.axis('off')
    plt.show()

## Start training

In [ ]:
trainer = FoodClassifier(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device
)

trainer.run_training(num_epochs=EPOCHS,save_path=DRIVE_BASE)

## Show the predict and true lables for foods

In [ ]:
# graphes
plot_training_results(trainer.get_history())
# visualize
visualize_results(model, test_dataset)

## Create UI

In [ ]:
%%writefile app.py
# --- Get API key via Colab Secrets ---
try:

    api_key_from_colab = userdata.get("GROQ_API_KEY")
    print(api_key_from_colab)
except:
    api_key_from_colab = None
    print("Please enter your API Key in the sidebar!")


#Basic Configuration ---
st.set_page_config(page_title="Food Guide", page_icon="🍣")

# Sidebar for API Key input to ensure privacy
with st.sidebar:
    st.title("🛠️ Settings")
    # set colab key as default, otherwise set api key as None

    api_key = st.text_input("Enter Groq API Key",
                            value=api_key_from_colab if api_key_from_colab else "",
                            type="password")
    print(api_key)
    if not api_key:
        st.info("Get your key at [console.groq.com](https://console.groq.com/)")



# --- 2. Llama Core Function ---
def ask_llama_chef(food_name, api_key):
    if not api_key:
        return "❌ Please enter your API Key in the sidebar!"

    try:
        client = Groq(api_key=api_key)
        prompt = f"""
        You are an expert travel guide serving a tourist from Canada.
        The vision model has identified this dish as "{food_name}".
        Please provide information in English: 1. Taste & Texture, 2. Travel Trivia, 3. Advice for Travelers.
        Tone: Humorous and friendly. Keep it under 150 words.
        """
        completion = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7,
        )
        return completion.choices[0].message.content
    except Exception as e:
        return f"Error occurred: {str(e)}"

# --- 3. UI Interface ---
st.title("🍣 AI Japanese Food Guide")
mock_predicted_label = st.selectbox(
    "Select a recognition result to test:",
    ["Sushi", "Ramen", "Takoyaki", "Tempura", "Okonomiyaki"]
)

if st.button("View Food Guide"):
    with st.spinner("Thinking..."):
        guide_text = ask_llama_chef(mock_predicted_label, api_key)
        st.chat_message("assistant", avatar="👨‍🍳").write(guide_text)

## Setup for Streamlit

In [ ]:
# install Python module
!pip install streamlit groq
# dlownload Cloudflare tunnle tool
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

## Activate Tunnel

In [ ]:
import subprocess
import time
import socket

# --- 1. Check if Streamlit is already running on port 8501 ---
def is_port_open(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('localhost', port)) == 0

# --- 2. Start Streamlit if it's not already running ---
if not is_port_open(8501):
    print("🚀 Starting Streamlit in the background...")
    subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.address", "0.0.0.0"],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5) # Give it time to boot
else:
    print("✅ Streamlit is already running.")

# --- 3. Start Cloudflare Tunnel and FORCE print logs ---
print("🌐 Opening Cloudflare Tunnel... (Look for the '.trycloudflare.com' link below)")
print("-" * 50)

# We use stdbuf to disable buffering so the URL appears immediately
p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8501"],
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

# This loop will print EVERYTHING Cloudflare says until the URL appears
for line in p.stdout:
    print(f"DEBUG: {line.strip()}") # This helps us see if there's an error
    if "trycloudflare.com" in line:
        url = line.strip().split(" ")[-1]
        print("\n" + "★" * 50)
        print(f"🔥 SUCCESS! YOUR APP IS LIVE AT:")
        print(f"👉 {url}")
        print("★" * 50)
        # We don't break, so the tunnel stays active in this cell